# Emotion Recognition from Facial Expressions
**Cognitive Science Course Project**

This notebook trains a CNN to classify 7 emotions from the FER2013 dataset:
Angry, Disgust, Fear, Happy, Sad, Surprise, Neutral

Grounded in Paul Ekman's theory of universal emotions.

## 1. Install & Import

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

## 2. Load the FER2013 Dataset

In [ ]:
import os

EMOTIONS = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

TRAIN_PATH = '/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/train.csv'
TEST_PATH  = '/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/test.csv'

df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

print('Train shape:', df_train.shape)
print('Test shape: ', df_test.shape)
print('Columns:', df_train.columns.tolist())
df_train.head()

## 3. Explore the Data

In [ ]:
# Emotion distribution
plt.figure(figsize=(10, 4))
counts = df_train['emotion'].value_counts().sort_index()
plt.bar(EMOTIONS, counts.values, color='steelblue')
plt.title('Emotion Distribution in Training Set')
plt.ylabel('Number of Images')
plt.xlabel('Emotion')
plt.tight_layout()
plt.show()
print(counts)

In [ ]:
# Sample faces per emotion
fig, axes = plt.subplots(1, 7, figsize=(16, 3))
for i, emotion in enumerate(EMOTIONS):
    sample = df_train[df_train['emotion'] == i].iloc[0]['pixels']
    img = np.array(sample.split(), dtype=np.uint8).reshape(48, 48)
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(emotion, fontsize=9)
    axes[i].axis('off')
plt.suptitle('One sample per emotion class')
plt.tight_layout()
plt.show()

## 4. Preprocess

In [ ]:
def parse_pixels(df):
    X = np.array([
        np.array(row.split(), dtype=np.float32).reshape(48, 48, 1) / 255.0
        for row in df['pixels']
    ])
    y = tf.keras.utils.to_categorical(df['emotion'], num_classes=7)
    return X, y

X_train, y_train = parse_pixels(df_train)

# test.csv from this competition has no emotion labels — use 20% of train for validation
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

print(f'Train: {len(X_train)} | Val: {len(X_val)}')

## 5. Build the CNN Model

In [ ]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(48, 48, 1)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.4),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(7, activation='softmax'),
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 6. Train

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_data=(X_val, y_val),
)

## 7. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Accuracy over Epochs')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Loss over Epochs')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.show()

## 8. Confusion Matrix

In [ ]:
y_pred = model.predict(X_val).argmax(axis=1)
y_true = y_val.argmax(axis=1)
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=EMOTIONS, yticklabels=EMOTIONS)
plt.title('Confusion Matrix — Which emotions get confused?')
plt.ylabel('True Emotion')
plt.xlabel('Predicted Emotion')
plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=EMOTIONS))

## 9. Save the Model

In [ ]:
model.save('emotion_model.h5')
print('Model saved — download it from the Output tab on the right')